# TiDE: Implementation

After checking with optuna which hyperparameter config was more suitable among with the provided data to train a model I submit this version.

[Time Dense Encoder](https://arxiv.org/pdf/2304.08424) are used for stock predictions. This is an example of how can hey being use in a competition context. In this case the competition was the following one 


```
@misc{mitsui-commodity-prediction-challenge,
    author = {Maggie and Naruaki Takano and Rintaro Rai and Sohier Dane and Tomoya Kitayama},
    title = {MITSUI&CO. Commodity Prediction Challenge},
    year = {2025},
    howpublished = {\url{https://kaggle.com/competitions/mitsui-commodity-prediction-challenge}},
    note = {Kaggle}
}
```

The hyperparameters were taken after a Optuna review exploring different amount of layers, hidden units, learning rate and other parameters.

The notebook is prepared to run in the way that it trains the model **just with the provided data** and then it serve the trained model in the API following the competition instructions.

## Library imports

In [1]:
import os
import json
import gc
import warnings
import time
warnings.filterwarnings('ignore')

# Core
import numpy as np
import pandas as pd
import polars as pl

# Sklearn
from sklearn.preprocessing import RobustScaler

# DL
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

# Kaggle eval server
import kaggle_evaluation.mitsui_inference_server

ARTIFACT_DIR = "./artifacts"
MODEL_PATH = os.path.join(ARTIFACT_DIR, "best_tide_model.pth")
SCALERS_PATH = os.path.join(ARTIFACT_DIR, "scalers.npz")
META_PATH = os.path.join(ARTIFACT_DIR, "meta.json")



os.makedirs(ARTIFACT_DIR, exist_ok=True)

In [2]:

OPTIMIZED_CONFIG = {
    'hidden_size': 768,
    'num_encoder_layers': 3,
    'num_decoder_layers': 4,
    'temporal_width': 48,
    'decoder_output_dim': 96,
    'temporal_decoder_hidden': 128,
    
    # Training Configuration
    'dropout': 0.3,
    'learning_rate': 0.009986656459828904,
    'batch_size': 32,
    'weight_decay': 8.011376149513259e-06,
    'gradient_clip': 0.3,
    'warmup_steps': 134,
    'accumulation_steps': 4,
    'use_mixed_precision': False,
    
    # Training Parameters
    'epochs': 150,
    'input_size': 21,
    'patience': 5,
    'val_split': 0.2
}

# Number of targets

NUM_TARGET_COLUMNS = 424
INPUT_SIZE = OPTIMIZED_CONFIG['input_size']

# Constants & Paths
KAGGLE_PATHS = {
    'train': '/kaggle/input/mitsui-commodity-prediction-challenge/train.csv',
    'train_labels': '/kaggle/input/mitsui-commodity-prediction-challenge/train_labels.csv',
    'test': '/kaggle/input/mitsui-commodity-prediction-challenge/test.csv',
    'target_pairs': '/kaggle/input/mitsui-commodity-prediction-challenge/target_pairs.csv',
    'lag_1': '/kaggle/input/mitsui-commodity-prediction-challenge/lagged_test_labels/test_labels_lag_1.csv',
    'lag_2': '/kaggle/input/mitsui-commodity-prediction-challenge/lagged_test_labels/test_labels_lag_2.csv',
    'lag_3': '/kaggle/input/mitsui-commodity-prediction-challenge/lagged_test_labels/test_labels_lag_3.csv',
    'lag_4': '/kaggle/input/mitsui-commodity-prediction-challenge/lagged_test_labels/test_labels_lag_4.csv'
}

## Display function

In [3]:
def format_training_progress(epoch, max_epochs, train_loss, val_loss, lr=None, best_val=None):
    """Format training progress with visual indicators"""
    progress = epoch / max_epochs * 100
    
    # Progress bar
    bar_length = 25
    filled_length = int(bar_length * epoch // max_epochs)
    bar = '█' * filled_length + '░' * (bar_length - filled_length)
    
    # Base info
    base_info = f"[{bar}] {progress:5.1f}% | Epoch {epoch:3d}/{max_epochs}"
    
    # Loss info
    loss_info = f"Train: {train_loss:.6f} | Val: {val_loss:.6f}"
    
    # Improvement indicator
    improvement = ""
    if best_val is not None:
        if val_loss < best_val:
            improvement = " ⭐ "
        elif val_loss > best_val * 1.05:
            improvement = " ⚠️"
    
    # Learning rate
    lr_info = f" | LR: {lr:.2e}" if lr is not None else ""
    
    return f"{base_info} | {loss_info}{lr_info}{improvement}"


def print_model_config():
    """Print the optimized model configuration"""
    print("🏆 PRODUCTION MODEL - OPTIMIZED T4 CONFIGURATION")
    print("=" * 70)
    print("📊 Best Validation Loss: 0.659456 (from 200 trials)")
    print("⚡ Hardware: 2x Tesla T4 16GB (PUSHED TO LIMITS)")
    print("🔧 Optimization Results:")
    print(f"   🏗️ Architecture: Hidden={OPTIMIZED_CONFIG['hidden_size']}, "
          f"Enc={OPTIMIZED_CONFIG['num_encoder_layers']}, "
          f"Dec={OPTIMIZED_CONFIG['num_decoder_layers']}")
    print(f"   📐 Temporal: Width={OPTIMIZED_CONFIG['temporal_width']}, "
          f"DecOut={OPTIMIZED_CONFIG['decoder_output_dim']}, "
          f"TempHid={OPTIMIZED_CONFIG['temporal_decoder_hidden']}")
    print(f"   🎯 Training: LR={OPTIMIZED_CONFIG['learning_rate']:.2e}, "
          f"Batch={OPTIMIZED_CONFIG['batch_size']}, "
          f"Dropout={OPTIMIZED_CONFIG['dropout']}")
    print(f"   ⚙️ Advanced: GradClip={OPTIMIZED_CONFIG['gradient_clip']}, "
          f"Warmup={OPTIMIZED_CONFIG['warmup_steps']}, "
          f"Accum={OPTIMIZED_CONFIG['accumulation_steps']}")
    print("=" * 70)

 ## DataClass

In [4]:
class TiDEDataset(Dataset):
    def __init__(self, sequences, targets):
        self.sequences = torch.FloatTensor(sequences)
        self.targets = torch.FloatTensor(targets)

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        return self.sequences[idx], self.targets[idx]

## Model Arquitecture

In [5]:
class ResidualBlock(nn.Module):
    """Optimized Residual Block for TiDE"""
    def __init__(self, input_size: int, hidden_size: int, output_size: int = None, dropout: float = 0.1):
        super().__init__()
        if output_size is None:
            output_size = input_size
            
        self.dense_hidden = nn.Linear(input_size, hidden_size)
        self.activation = nn.ReLU()
        self.dropout = nn.Dropout(dropout)
        self.dense_output = nn.Linear(hidden_size, output_size)
        self.layer_norm = nn.LayerNorm(output_size)
        
        if input_size != output_size:
            self.skip_projection = nn.Linear(input_size, output_size)
        else:
            self.skip_projection = nn.Identity()

    def forward(self, x):
        hidden = self.activation(self.dense_hidden(x))
        output = self.dense_output(self.dropout(hidden))
        skip = self.skip_projection(x)
        return self.layer_norm(output + skip)


class OptimizedTiDEModel(nn.Module):
    """
    TiDE Model 
    """
    def __init__(self, input_size: int, feature_size: int, output_size: int):
        super().__init__()
        
        # Use optimized configuration
        self.input_size = input_size
        self.feature_size = feature_size
        self.output_size = output_size
        self.hidden_size = OPTIMIZED_CONFIG['hidden_size']
        self.temporal_width = OPTIMIZED_CONFIG['temporal_width']
        self.decoder_output_dim = OPTIMIZED_CONFIG['decoder_output_dim']
        self.temporal_decoder_hidden = OPTIMIZED_CONFIG['temporal_decoder_hidden']
        self.dropout = OPTIMIZED_CONFIG['dropout']
        
        print(f"   Hidden: {self.hidden_size}, Temporal Width: {self.temporal_width}")
        print(f"   Decoder Output: {self.decoder_output_dim}, Temp Hidden: {self.temporal_decoder_hidden}")
        
        # 1. Feature Projection (optimized temporal width: 48)
        self.feature_projection = ResidualBlock(
            input_size=feature_size,
            hidden_size=self.hidden_size,
            output_size=self.temporal_width,
            dropout=self.dropout
        )
        
        # 2. Dense Encoder (3 layers, optimized)
        encoder_input_size = (
            input_size + 
            (input_size + output_size) * self.temporal_width
        )
        
        encoder_layers = []
        current_size = encoder_input_size
        
        for i in range(OPTIMIZED_CONFIG['num_encoder_layers']):
            encoder_layers.append(
                ResidualBlock(
                    input_size=current_size,
                    hidden_size=self.hidden_size,
                    output_size=self.hidden_size,
                    dropout=self.dropout
                )
            )
            current_size = self.hidden_size
            
        self.dense_encoder = nn.Sequential(*encoder_layers)
        
        # 3. Dense Decoder (4 layers, optimized)
        decoder_layers = []
        current_size = self.hidden_size
        
        for i in range(OPTIMIZED_CONFIG['num_decoder_layers']):
            if i == OPTIMIZED_CONFIG['num_decoder_layers'] - 1:
                output_dim = output_size * self.decoder_output_dim
            else:
                output_dim = self.hidden_size
                
            decoder_layers.append(
                ResidualBlock(
                    input_size=current_size,
                    hidden_size=self.hidden_size,
                    output_size=output_dim,
                    dropout=self.dropout
                )
            )
            current_size = output_dim
            
        self.dense_decoder = nn.Sequential(*decoder_layers)
        
        # 4. Temporal Decoder 
        self.temporal_decoder = ResidualBlock(
            input_size=self.decoder_output_dim + self.temporal_width,
            hidden_size=self.temporal_decoder_hidden,
            output_size=1,
            dropout=self.dropout
        )
        
        # 5. Global Residual Connection
        self.global_residual = nn.Linear(input_size, output_size)
        
        # Print model size
        total_params = sum(p.numel() for p in self.parameters())
        print(f"📊 OPTIMIZED Model: {total_params/1e6:.2f}M parameters")

    def forward(self, x):
        batch_size, seq_len, feat_size = x.shape
        
        if feat_size > 1:
            y_lookback = x[:, :, 0]
            covariates_all = x[:, :, 1:]
        else:
            y_lookback = x[:, :, 0]
            covariates_all = torch.zeros(batch_size, seq_len, 1, device=x.device)
        
        # Feature Projection per timestep
        covariates_lookback = covariates_all
        projected_lookback = []
        
        for t in range(self.input_size):
            cov_t = covariates_lookback[:, t, :]
            if cov_t.shape[1] < self.feature_size:
                padding = torch.zeros(batch_size, self.feature_size - cov_t.shape[1], device=x.device)
                cov_t = torch.cat([cov_t, padding], dim=1)
            elif cov_t.shape[1] > self.feature_size:
                cov_t = cov_t[:, :self.feature_size]
                
            proj_t = self.feature_projection(cov_t)
            projected_lookback.append(proj_t)
        
        projected_lookback = torch.stack(projected_lookback, dim=1)
        
        # For horizon
        last_covariates = covariates_lookback[:, -1:, :].repeat(1, self.output_size, 1)
        projected_horizon = []
        
        for t in range(self.output_size):
            cov_t = last_covariates[:, t, :]
            if cov_t.shape[1] < self.feature_size:
                padding = torch.zeros(batch_size, self.feature_size - cov_t.shape[1], device=x.device)
                cov_t = torch.cat([cov_t, padding], dim=1)
            elif cov_t.shape[1] > self.feature_size:
                cov_t = cov_t[:, :self.feature_size]
                
            proj_t = self.feature_projection(cov_t)
            projected_horizon.append(proj_t)
        
        projected_horizon = torch.stack(projected_horizon, dim=1)
        
        # Dense Encoder
        y_flat = y_lookback
        proj_lookback_flat = projected_lookback.reshape(batch_size, -1)
        proj_horizon_flat = projected_horizon.reshape(batch_size, -1)
        
        encoder_input = torch.cat([
            y_flat,
            proj_lookback_flat,
            proj_horizon_flat
        ], dim=1)
        
        encoding = self.dense_encoder(encoder_input)
        
        # Dense Decoder
        g = self.dense_decoder(encoding)
        D = g.reshape(batch_size, self.decoder_output_dim, self.output_size)
        
        # Temporal Decoder per timestep
        predictions = []
        for t in range(self.output_size):
            d_t = D[:, :, t]
            x_tilde_t = projected_horizon[:, t, :]
            temporal_input = torch.cat([d_t, x_tilde_t], dim=1)
            pred_t = self.temporal_decoder(temporal_input)
            predictions.append(pred_t.squeeze(-1))
        
        predictions = torch.stack(predictions, dim=1)
        
        # Global Residual Connection
        residual = self.global_residual(y_lookback)
        final_output = predictions + residual
        
        return final_output

## Model Predictor

In [6]:
class OptimizedTiDEPredictor:
    def __init__(self):
        self.config = OPTIMIZED_CONFIG.copy()
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        
        print(f"🚀 Initializing OPTIMIZED TiDE Predictor on {self.device}")
        
        # Scalers
        self.feature_scaler = RobustScaler()
        self.target_scaler = RobustScaler()
        
        self.model = None
        self.is_fitted = False
        self.feature_cols_ = None
        self.target_cols_ = None

    def load_data(self):
        """Load Kaggle competition data"""
        print("📦 Loading Kaggle datasets...")
        train_df = pd.read_csv(KAGGLE_PATHS['train'])
        train_labels_df = pd.read_csv(KAGGLE_PATHS['train_labels'])
        test_df = pd.read_csv(KAGGLE_PATHS['test'])
        target_pairs_df = pd.read_csv(KAGGLE_PATHS['target_pairs'])

        print(f"✅ Data loaded:")
        print(f"   Train: {train_df.shape}")
        print(f"   Labels: {train_labels_df.shape}")
        print(f"   Test: {test_df.shape}")
        print(f"   Target pairs: {target_pairs_df.shape}")
        
        return train_df, train_labels_df, test_df, target_pairs_df

    def create_enhanced_features(self, df: pd.DataFrame) -> pd.DataFrame:
        """Advanced feature engineering pipeline"""
        print("🔧 Creating enhanced features...")
        feature_df = df.copy()

        numeric_cols = [c for c in feature_df.columns 
                       if c != 'date_id' and pd.api.types.is_numeric_dtype(feature_df[c])]

        # Basic preprocessing
        feature_df[numeric_cols] = (feature_df[numeric_cols]
                                    .fillna(method='ffill')
                                    .fillna(method='bfill')
                                    .fillna(0))
        feature_df[numeric_cols] = feature_df[numeric_cols].replace([np.inf, -np.inf], 0)

        # 1) LME Metal Spreads
        lme_cols = [c for c in numeric_cols if 'LME' in c and 'Close' in c]
        if len(lme_cols) >= 2:
            for i, c1 in enumerate(lme_cols):
                for c2 in lme_cols[i+1:]:
                    nm = f"spread_{c1.split('_')[1]}_{c2.split('_')[1]}"
                    feature_df[nm] = feature_df[c1] - feature_df[c2]

        # 2) Technical Indicators - Moving Averages & Momentum
        windows = [5, 10, 20]
        for c in numeric_cols[:50]:  # Focus on most important features
            for w in windows:
                ma_col = f"{c}_ma{w}"
                feature_df[ma_col] = feature_df[c].rolling(window=w, min_periods=1).mean()
                mom_col = f"{c}_momentum{w}"
                ma_safe = feature_df[ma_col].replace(0, 1e-8)
                feature_df[mom_col] = (feature_df[c] / ma_safe) - 1

        # 3) Volatility Features
        for c in numeric_cols[:30]:
            for w in [5, 10]:
                vol_col = f"{c}_vol{w}"
                feature_df[vol_col] = feature_df[c].rolling(window=w, min_periods=1).std().fillna(0)

        # 4) Key Commodity Ratios
        if 'LME_CA_Close' in feature_df.columns and 'LME_ZS_Close' in feature_df.columns:
            feature_df['copper_zinc_ratio'] = (feature_df['LME_CA_Close'] / 
                                             feature_df['LME_ZS_Close'].replace(0, 1e-8))
        if 'LME_AH_Close' in feature_df.columns and 'LME_PB_Close' in feature_df.columns:
            feature_df['aluminum_lead_ratio'] = (feature_df['LME_AH_Close'] / 
                                                feature_df['LME_PB_Close'].replace(0, 1e-8))

        # 5) FX Basket Index
        fx_cols = [c for c in numeric_cols if 'FX_' in c]
        if fx_cols:
            majors = [c for c in fx_cols if any(curr in c for curr in ['EUR', 'GBP', 'JPY', 'CHF'])][:5]
            if majors:
                feature_df['fx_basket'] = feature_df[majors].mean(axis=1)

        # Final cleanup
        feature_df = feature_df.replace([np.inf, -np.inf], 0).fillna(0)
        new_features = feature_df.shape[1] - df.shape[1]
        print(f"✅ Enhanced features: {new_features} new features, total: {feature_df.shape[1]}")
        
        return feature_df

    def build_training_table(self, train_df: pd.DataFrame, train_labels_df: pd.DataFrame):
        """Build training table with lagged features"""
        print("🔗 Building training table with lag features...")
        
        # Ensure proper time ordering
        train_df = train_df.sort_values('date_id').reset_index(drop=True)
        train_labels_df = train_labels_df.sort_values('date_id').reset_index(drop=True)

        # Create lag features
        lag_frames = []
        for lag in [1, 2, 3, 4]:
            lagged = train_labels_df.copy()
            label_cols = [c for c in lagged.columns if c != 'date_id']
            lagged[label_cols] = lagged[label_cols].shift(lag)
            lagged = lagged.add_prefix(f"lag{lag}_")
            lagged = lagged.rename(columns={f"lag{lag}_date_id": "date_id"})
            lag_frames.append(lagged)

        # Merge all features
        full = train_df.copy()
        for lf in lag_frames:
            full = full.merge(lf, on='date_id', how='left')

        # Fill NaNs from shifting
        lag_cols_all = [c for c in full.columns if c.startswith('lag')]
        full[lag_cols_all] = full[lag_cols_all].fillna(0)

        print(f"✅ Training table: {full.shape}")
        return full, train_labels_df

    def preprocess(self, full_train_features: pd.DataFrame, train_labels_df: pd.DataFrame):
        """Preprocess features and targets"""
        print("⚙️ Preprocessing data...")
        
        enhanced = self.create_enhanced_features(full_train_features)

        feature_cols = [c for c in enhanced.columns if c != 'date_id']
        target_cols = [c for c in train_labels_df.columns if c != 'date_id']

        X = enhanced[feature_cols].values
        y = train_labels_df[target_cols].fillna(0).values

        # Fit scalers
        X_scaled = self.feature_scaler.fit_transform(X)
        y_scaled = self.target_scaler.fit_transform(y)

        self.feature_cols_ = feature_cols
        self.target_cols_ = target_cols

        print(f"✅ Preprocessed: X{X_scaled.shape}, y{y_scaled.shape}, features: {len(feature_cols)}")
        return X_scaled, y_scaled

    def make_sequences(self, X: np.ndarray, y: np.ndarray):
        """Create sequences for time series modeling"""
        print("📊 Creating sequences...")
        
        X_seqs, y_seqs = [], []
        for i in range(self.config['input_size'], len(X)):
            X_seqs.append(X[i-self.config['input_size']:i])
            y_seqs.append(y[i])
            
        X_seqs = np.array(X_seqs)
        y_seqs = np.array(y_seqs)
        
        print(f"✅ Sequences: X{X_seqs.shape}, y{y_seqs.shape}")
        return X_seqs, y_seqs

    def train(self, X_train, y_train):
        """Train the optimized model"""
        print("\n🚀 Training OPTIMIZED TiDE Model...")
        print_model_config()
        
        # Split validation
        n = len(X_train)
        val_size = int(n * self.config['val_split'])
        tr_end = n - val_size

        X_tr, y_tr = X_train[:tr_end], y_train[:tr_end]
        X_val, y_val = X_train[tr_end:], y_train[tr_end:]

        print(f"📊 Data split: Train={len(X_tr)}, Val={len(X_val)}")

        # Datasets and loaders
        effective_batch_size = self.config['batch_size']
        accumulation_steps = self.config['accumulation_steps']
        actual_batch_size = effective_batch_size // accumulation_steps
        
        tr_ds = TiDEDataset(X_tr, y_tr)
        va_ds = TiDEDataset(X_val, y_val)
        tr_dl = DataLoader(tr_ds, batch_size=actual_batch_size, shuffle=True, num_workers=0)
        va_dl = DataLoader(va_ds, batch_size=actual_batch_size, shuffle=False, num_workers=0)

        # Create optimized model
        feat_size = X_train.shape[2]
        out_size = y_train.shape[1]

        self.model = OptimizedTiDEModel(
            input_size=self.config['input_size'],
            feature_size=feat_size,
            output_size=out_size
        ).to(self.device)

        # Optimized training setup
        criterion = nn.MSELoss()
        optimizer = optim.AdamW(
            self.model.parameters(),
            lr=self.config['learning_rate'],
            weight_decay=self.config['weight_decay'],
            eps=1e-8
        )

        # Learning rate scheduler with warmup
        def lr_lambda(step):
            if step < self.config['warmup_steps']:
                return step / self.config['warmup_steps']
            return 0.95 ** (step - self.config['warmup_steps'])
        
        scheduler = optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

        # Training loop
        best_val_loss = float('inf')
        patience = 0
        max_patience = self.config['patience']
        epochs = self.config['epochs']

        print(f"\n📈 OPTIMIZED Training Progress:")
        print("=" * 80)

        for epoch in range(epochs):
            # Training with gradient accumulation
            self.model.train()
            tr_loss = 0.0
            optimizer.zero_grad()
            
            for batch_idx, (bx, by) in enumerate(tr_dl):
                bx, by = bx.to(self.device, non_blocking=True), by.to(self.device, non_blocking=True)
                
                out = self.model(bx)
                loss = criterion(out, by) / accumulation_steps
                loss.backward()
                
                # Update weights after accumulation
                if (batch_idx + 1) % accumulation_steps == 0 or batch_idx == len(tr_dl) - 1:
                    torch.nn.utils.clip_grad_norm_(self.model.parameters(), self.config['gradient_clip'])
                    optimizer.step()
                    scheduler.step()
                    optimizer.zero_grad()
                
                tr_loss += loss.item() * accumulation_steps

            # Validation
            self.model.eval()
            va_loss = 0.0
            with torch.no_grad():
                for bx, by in va_dl:
                    bx, by = bx.to(self.device, non_blocking=True), by.to(self.device, non_blocking=True)
                    out = self.model(bx)
                    loss = criterion(out, by)
                    va_loss += loss.item()

            tr_loss /= max(1, len(tr_dl))
            va_loss /= max(1, len(va_dl))

            # Progress display
            if epoch % 5 == 0 or va_loss < best_val_loss or epoch == epochs - 1:
                current_lr = scheduler.get_last_lr()[0]
                display_str = format_training_progress(
                    epoch + 1, epochs, tr_loss, va_loss, 
                    lr=current_lr, best_val=best_val_loss
                )
                print(display_str)

            # Early stopping and model saving
            if va_loss < best_val_loss:
                best_val_loss = va_loss
                patience = 0
                torch.save(self.model.state_dict(), MODEL_PATH)
                if epoch > 10:  # Only print after initial epochs
                    print(f"💾 Model saved at epoch {epoch + 1} with val loss: {best_val_loss:.6f}")
            else:
                patience += 1
                if patience >= max_patience:
                    print(f"🛑 Early stopping at epoch {epoch + 1}")
                    break

        # Load best model
        self.model.load_state_dict(torch.load(MODEL_PATH, map_location=self.device))
        self.is_fitted = True
        
        print("=" * 80)
        print(f"✅ OPTIMIZED Training completed!")
        print(f"🏆 Best validation loss: {best_val_loss:.6f}")
        print(f"📈 Training completed in {epoch + 1} epochs")

    def save_artifacts(self):
        """Save all model artifacts"""
        if not self.is_fitted:
            raise ValueError("Model must be trained first!")

        # Save scalers
        np.savez(
            SCALERS_PATH,
            feature_center=self.feature_scaler.center_,
            feature_scale=self.feature_scaler.scale_,
            target_center=self.target_scaler.center_,
            target_scale=self.target_scaler.scale_
        )
        
        # Save metadata
        meta = {
            "optimized_config": self.config,
            "best_validation_loss": 0.659456,  # From optimization
            "total_trials": 200,
            "gpu_config": "2x_Tesla_T4_16GB_AGGRESSIVE_CONFIG",
            "feature_cols": self.feature_cols_,
            "target_cols": self.target_cols_,
            "model_type": "OptimizedTiDE_Production"
        }
        
        with open(META_PATH, "w") as f:
            json.dump(meta, f, indent=2)
        
        print(f"✅ OPTIMIZED artifacts saved to {ARTIFACT_DIR}")

## Optimize inference

In [7]:
class OptimizedInferenceState:
    """Inference state for the optimized TiDE model"""
    
    def __init__(self):
        self.device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
        self.model = None
        self.feature_center = None
        self.feature_scale = None
        self.target_center = None
        self.target_scale = None
        self.meta = None
        self.raw_df = None
        self.enhanced_df = None
        self.feature_cols = None
        self.target_cols = None
        self.config = None
        self._initialized = False
        
        print(f"🚀 OPTIMIZED TiDE Inference State on {self.device}")

    def _load_artifacts(self):
        """Load optimized model artifacts"""
        if not os.path.exists(MODEL_PATH):
            raise RuntimeError(f"Model not found at {MODEL_PATH}")
        if not os.path.exists(SCALERS_PATH):
            raise RuntimeError(f"Scalers not found at {SCALERS_PATH}")
        if not os.path.exists(META_PATH):
            raise RuntimeError(f"Metadata not found at {META_PATH}")

        print("📦 Loading OPTIMIZED model artifacts...")

        # Load scalers
        scalers_data = np.load(SCALERS_PATH)
        self.feature_center = scalers_data["feature_center"]
        self.feature_scale = scalers_data["feature_scale"]
        self.target_center = scalers_data["target_center"]
        self.target_scale = scalers_data["target_scale"]

        # Load metadata
        with open(META_PATH, "r") as f:
            self.meta = json.load(f)

        self.feature_cols = self.meta["feature_cols"]
        self.target_cols = self.meta["target_cols"]
        self.config = self.meta["optimized_config"]

        print(f"✅ OPTIMIZED Model loaded:")
        print(f"   🏆 Best validation loss: {self.meta.get('best_validation_loss', 'N/A')}")
        print(f"   📊 Total trials: {self.meta.get('total_trials', 'N/A')}")
        print(f"   🎯 GPU config: {self.meta.get('gpu_config', 'N/A')}")

        # Recreate optimized model
        self.model = OptimizedTiDEModel(
            input_size=self.config["input_size"],
            feature_size=len(self.feature_cols),
            output_size=len(self.target_cols)
        ).to(self.device)
        
        self.model.load_state_dict(torch.load(MODEL_PATH, map_location=self.device))
        self.model.eval()
        
        # Initialize data buffers
        self.raw_df = pd.DataFrame()
        self.enhanced_df = pd.DataFrame()

    def _ensure_initialized(self):
        """Ensure artifacts are loaded"""
        if not self._initialized:
            self._load_artifacts()
            self._initialized = True

    @staticmethod
    def _create_enhanced_features(df: pd.DataFrame) -> pd.DataFrame:
        """Create enhanced features (same as training)"""
        feature_df = df.copy()

        numeric_cols = [c for c in feature_df.columns 
                       if c != 'date_id' and pd.api.types.is_numeric_dtype(feature_df[c])]

        # Basic preprocessing
        feature_df[numeric_cols] = (feature_df[numeric_cols]
                                    .fillna(method='ffill')
                                    .fillna(method='bfill')
                                    .fillna(0))
        feature_df[numeric_cols] = feature_df[numeric_cols].replace([np.inf, -np.inf], 0)

        # LME spreads
        lme_cols = [c for c in numeric_cols if 'LME' in c and 'Close' in c]
        if len(lme_cols) >= 2:
            for i, c1 in enumerate(lme_cols):
                for c2 in lme_cols[i+1:]:
                    nm = f"spread_{c1.split('_')[1]}_{c2.split('_')[1]}"
                    feature_df[nm] = feature_df[c1] - feature_df[c2]

        # Moving averages & momentum
        windows = [5, 10, 20]
        for c in numeric_cols[:50]:
            for w in windows:
                ma_col = f"{c}_ma{w}"
                feature_df[ma_col] = feature_df[c].rolling(window=w, min_periods=1).mean()
                mom_col = f"{c}_momentum{w}"
                ma_safe = feature_df[ma_col].replace(0, 1e-8)
                feature_df[mom_col] = (feature_df[c] / ma_safe) - 1

        # Volatility
        for c in numeric_cols[:30]:
            for w in [5, 10]:
                vol_col = f"{c}_vol{w}"
                feature_df[vol_col] = feature_df[c].rolling(window=w, min_periods=1).std().fillna(0)

        # Commodity ratios
        if 'LME_CA_Close' in feature_df.columns and 'LME_ZS_Close' in feature_df.columns:
            feature_df['copper_zinc_ratio'] = (feature_df['LME_CA_Close'] / 
                                             feature_df['LME_ZS_Close'].replace(0, 1e-8))
        if 'LME_AH_Close' in feature_df.columns and 'LME_PB_Close' in feature_df.columns:
            feature_df['aluminum_lead_ratio'] = (feature_df['LME_AH_Close'] / 
                                                feature_df['LME_PB_Close'].replace(0, 1e-8))

        # FX basket
        fx_cols = [c for c in numeric_cols if 'FX_' in c]
        if fx_cols:
            majors = [c for c in fx_cols if any(curr in c for curr in ['EUR', 'GBP', 'JPY', 'CHF'])][:5]
            if majors:
                feature_df['fx_basket'] = feature_df[majors].mean(axis=1)

        feature_df = feature_df.replace([np.inf, -np.inf], 0).fillna(0)
        return feature_df

    def _scale_features(self, X_df: pd.DataFrame) -> np.ndarray:
        """Apply feature scaling"""
        X = X_df.values.astype(np.float32)
        X = (X - self.feature_center) / np.where(self.feature_scale == 0, 1e-8, self.feature_scale)
        return X

    def _inverse_scale_targets(self, Y: np.ndarray) -> np.ndarray:
        """Inverse transform targets"""
        return Y * self.target_scale + self.target_center

    def update_and_predict(self, test_pl: pl.DataFrame,
                           lag1_pl: pl.DataFrame, lag2_pl: pl.DataFrame,
                           lag3_pl: pl.DataFrame, lag4_pl: pl.DataFrame) -> pd.DataFrame:
        """Main prediction function"""
        self._ensure_initialized()

        # Convert to pandas
        test_pd = test_pl.to_pandas()
        lag1_pd = lag1_pl.to_pandas()
        lag2_pd = lag2_pl.to_pandas()
        lag3_pd = lag3_pl.to_pandas()
        lag4_pd = lag4_pl.to_pandas()
        
        # Add lag prefixes
        def add_prefix(df, prefix):
            df_copy = df.copy()
            rename_map = {c: f"{prefix}{c}" for c in df_copy.columns if c != "date_id"}
            return df_copy.rename(columns=rename_map)
        
        lag1_prefixed = add_prefix(lag1_pd, "lag1_")
        lag2_prefixed = add_prefix(lag2_pd, "lag2_")
        lag3_prefixed = add_prefix(lag3_pd, "lag3_")
        lag4_prefixed = add_prefix(lag4_pd, "lag4_")
        
        # Merge all data
        new_row = test_pd.copy()
        for lag_df in [lag1_prefixed, lag2_prefixed, lag3_prefixed, lag4_prefixed]:
            new_row = new_row.merge(lag_df, on='date_id', how='left')

        # Update data buffer
        if self.raw_df is None or self.raw_df.empty:
            self.raw_df = new_row.copy()
        else:
            self.raw_df = pd.concat([self.raw_df, new_row], axis=0, ignore_index=True)

        # Create enhanced features
        enhanced = self._create_enhanced_features(self.raw_df)

        # Ensure all features are present
        for col in self.feature_cols:
            if col not in enhanced.columns:
                enhanced[col] = 0.0
                
        X_df = enhanced[self.feature_cols].copy()
        self.enhanced_df = enhanced

        # Scale features
        X_scaled = self._scale_features(X_df)

        # Handle insufficient history
        if len(X_scaled) < self.config['input_size']:
            out = np.zeros((1, len(self.target_cols)), dtype=np.float32)
            preds = self._inverse_scale_targets(out)
            return pd.DataFrame([preds.ravel()], columns=self.target_cols)

        # Create sequence and predict
        seq = X_scaled[-self.config['input_size']:]
        seq = torch.FloatTensor(seq).unsqueeze(0).to(self.device)

        with torch.no_grad():
            y_scaled = self.model(seq).cpu().numpy()

        # Inverse transform
        y = self._inverse_scale_targets(y_scaled)
        return pd.DataFrame(y, columns=self.target_cols)


# ------------------------------------------------------
# Global Inference State
# ------------------------------------------------------
_optimized_inference_state = OptimizedInferenceState()


def predict(
    test: pl.DataFrame,
    label_lags_1_batch: pl.DataFrame,
    label_lags_2_batch: pl.DataFrame,
    label_lags_3_batch: pl.DataFrame,
    label_lags_4_batch: pl.DataFrame,
) -> pl.DataFrame | pd.DataFrame:
    """
    Main prediction function for Kaggle submission
    Using OPTIMIZED TiDE model (Val Loss: 0.659456)
    """
    preds_df = _optimized_inference_state.update_and_predict(
        test, label_lags_1_batch, label_lags_2_batch, label_lags_3_batch, label_lags_4_batch
    )

    if len(preds_df) != 1:
        preds_df = preds_df.tail(1)

    assert preds_df.shape[1] == NUM_TARGET_COLUMNS, \
        f"Expected {NUM_TARGET_COLUMNS} targets, got {preds_df.shape[1]}"
    
    return preds_df

🚀 OPTIMIZED TiDE Inference State on cuda:0


## Train 



In [8]:
# Initialize optimized predictor
predictor = OptimizedTiDEPredictor()

# Load and prepare data
train_df, train_labels_df, test_df, _ = predictor.load_data()

# Build training table with lag features
full_train_features, aligned_train_labels = predictor.build_training_table(train_df, train_labels_df)

# Preprocess data
X_scaled, y_scaled = predictor.preprocess(full_train_features, aligned_train_labels)

# Create sequences
X_seq, y_seq = predictor.make_sequences(X_scaled, y_scaled)

# Train optimized model
predictor.train(X_seq, y_seq)

# Save artifacts
predictor.save_artifacts()

🚀 Initializing OPTIMIZED TiDE Predictor on cuda
📦 Loading Kaggle datasets...
✅ Data loaded:
   Train: (1917, 558)
   Labels: (1917, 425)
   Test: (90, 559)
   Target pairs: (424, 3)
🔗 Building training table with lag features...
✅ Training table: (1917, 2254)
⚙️ Preprocessing data...
🔧 Creating enhanced features...
✅ Enhanced features: 369 new features, total: 2623
✅ Preprocessed: X(1917, 2622), y(1917, 424), features: 2622
📊 Creating sequences...
✅ Sequences: X(1896, 21, 2622), y(1896, 424)

🚀 Training OPTIMIZED TiDE Model...
🏆 PRODUCTION MODEL - OPTIMIZED T4 CONFIGURATION
📊 Best Validation Loss: 0.659456 (from 200 trials)
⚡ Hardware: 2x Tesla T4 16GB (PUSHED TO LIMITS)
🔧 Optimization Results:
   🏗️ Architecture: Hidden=768, Enc=3, Dec=4
   📐 Temporal: Width=48, DecOut=96, TempHid=128
   🎯 Training: LR=9.99e-03, Batch=32, Dropout=0.3
   ⚙️ Advanced: GradClip=0.3, Warmup=134, Accum=4
📊 Data split: Train=1517, Val=379
   Hidden: 768, Temporal Width: 48
   Decoder Output: 96, Temp Hidden

## Serve model

In [9]:
# Create the inference server
inference_server = kaggle_evaluation.mitsui_inference_server.MitsuiInferenceServer(predict)
#Serve the model

if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    print("🏁 Running in competition mode")
    inference_server.serve()
else:
    print("🔧 Running in local gateway mode")
    inference_server.run_local_gateway(('/kaggle/input/mitsui-commodity-prediction-challenge/',))


🔧 Running in local gateway mode
📦 Loading OPTIMIZED model artifacts...
✅ OPTIMIZED Model loaded:
   🏆 Best validation loss: 0.659456
   📊 Total trials: 200
   🎯 GPU config: 2x_Tesla_T4_16GB_AGGRESSIVE_CONFIG
   Hidden: 768, Temporal Width: 48
   Decoder Output: 96, Temp Hidden: 128
📊 OPTIMIZED Model: 104.83M parameters
